# 4章 データからコンテキストへ - RAG (2) :ベクトル埋め込み、再ランキング

## 事前準備

当サンプルコードを実行するには、OpenAIのAPIキーが必要です。

### OpenAI
- OpenAIのプラットフォーム（ https://platform.openai.com/ ）にアクセスし、ご自身のアカウントでログインして、APIキーを生成してください。
- 生成したAPIキーを定数 `OPENAI_API_KEY` に設定します。

```
OPENAI_API_KEY=<作成したAPIキー>
```


In [ ]:
OPENAI_API_KEY="<作成したAPIキー>"

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
!wget "https://drive.google.com/uc?export=download&id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_" -O apple.txt

--2026-10-22 05:55:13--  https://drive.google.com/uc?export=download&id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_
Resolving drive.google.com (drive.google.com)... 74.125.24.139, 74.125.24.101, 74.125.24.102, ...
Connecting to drive.google.com (drive.google.com)|74.125.24.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_&export=download [following]
--2026-10-22 05:55:13--  https://drive.usercontent.google.com/download?id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.24.132, 2404:6800:4003:c03::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.24.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 410321 (401K) [application/octet-stream]
Saving to: ‘apple.txt’

apple.txt           100%[===================>] 400.70K  --.-KB/s    in 0.003

In [ ]:
!pip install langchain_experimental langchain_huggingface langchain-litellm langchain_classic langchain_openai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 590.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 934.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate

def create_vector_store(text: str, collection_name: str = "10k_chunks"):
    """
    10-K報告書のテキストをチャンクに分割し、埋め込みを作成してChromaDBに保存する。

    Args:
        text (str): 10-Kの全文テキスト
        collection_name (str): ChromaDBコレクションの名前

    Returns:
        Chroma: 埋め込み済みチャンクを含むベクトルストア
    """
    # テキストをチャンクに分割する
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = text_splitter.split_text(text)

    # 埋め込みを作成し、ChromaDBに保存する
    # persist_directoryを指定すると、データがディスクに永続化され、後から再利用できる。
    vectorstore = Chroma.from_texts(
        texts=chunks,
        embedding=HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        ),
        collection_name=collection_name,
        persist_directory="./chroma_db"  # 後から再利用できるようにディスクに保存する
    )

    return vectorstore


/tmp/ipykernel_879/2923826189.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [ ]:
def retrieve_and_summarize_risks(vectorstore, top_k: int = 5):
    """
    リスク関連のチャンクを検索し、リスク要因の要約を生成する。

    Args:
        vectorstore: ChromaDBのベクトルストア
        top_k (int): 取得する最も関連性の高いチャンク数

    Returns:
        str: 生成されたリスク要因の要約
    """
    # リスク要因に関連するチャンクを検索する
    query = """risk factors, threats, vulnerabilities, regulatory concerns,
    market risks, operational risks"""

    # 関連性の高い上位K件のチャンクを取得する
    relevant_chunks = vectorstore.similarity_search(query, k=top_k)

    # 取得したチャンクを結合する
    combined_context = "\n\n---\n\n".join(
        [doc.page_content for doc in relevant_chunks]
    )

    # 要約用のプロンプトを作成する
    prompt_template = """
    ROLE: You are an expert financial analyst specializing
    in risk assessment for SEC filings.

    CONTEXT: Below are the most relevant sections from a
    10-K filing related to risk factors.

    INSTRUCTION: Create a comprehensive risk factors summary.
    Organize the risks into categories
    (Market Risks, Operational Risks, Regulatory Risks,
    Financial Risks, etc.) and highlight
    the most material risks to the company.

    RETRIEVED SECTIONS:
    {context}

    REQUIREMENTS:
    - Categorize risks logically
    - Prioritize material risks
    - Keep summary under 500 words
    - Use bullet points for clarity

    RISK FACTORS SUMMARY:
    """

    prompt = PromptTemplate(
        input_variables=["context"],
        template=prompt_template
    )

    llm = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)
    chain = LLMChain(llm=llm, prompt=prompt)

    summary = chain.run(context=combined_context)

    return summary

# サンプルの10-K SEC提出書類からテキストを読み込む
with open('apple.txt', 'r') as file:
    ten_k_text = file.read()

# ベクトルストアを作成して永続化する。この処理は1度のみ行う
vectorstore = create_vector_store(ten_k_text)

# リスク要約を生成する
risk_summary = retrieve_and_summarize_risks(vectorstore, top_k=5)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
risk_summary

'**Risk Factors Summary**\n\n---\n\n### 1. **Cybersecurity and Data Privacy Risks (Operational Risks)**\n- The Company faces frequent and increasingly sophisticated malicious cyberattacks aimed at compromising confidential information, disrupting operations, and damaging reputation.\n- High-profile status and valuable data increase the likelihood of being targeted.\n- Security measures (encryption, multifactor authentication, monitoring) may not fully prevent breaches caused by hacking, ransomware, employee error, or social engineering.\n- Cyber incidents can lead to customer loss, litigation, regulatory penalties, and stock price volatility.\n- Insurance coverage for cyber risks may be insufficient to cover all potential losses.\n\n### 2. **Market and Economic Risks**\n- Exposure to fluctuations in U.S. interest rates affects investment portfolio fair value and increases interest expense on term debt.\n- Foreign exchange rate volatility, especially a strengthening U.S. dollar, can neg

In [ ]:
from sentence_transformers import CrossEncoder

def rerank_with_cross_encoder(query: str, documents: list, top_k: int = 5):
    """
    クロスエンコーダーを使用して取得済み文書を再ランキングする。

    Args:
        query (str): 検索クエリ
        documents (list): 初回検索で取得されたLangChainのDocumentオブジェクトのリスト
        top_k (int): 再ランキング後に返す上位文書数

    Returns:
        list: 関連性が高い順に並べ替えられた文書
    """
    # クロスエンコーダーモデルを読み込む
    # ms-marco-MiniLMは、特にパッセージランキング用に学習されている
    cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')

    # スコアリング用にクエリと文書のペアを作成する
    pairs = [[query, doc.page_content] for doc in documents]

    # すべてのペアにスコアを付ける
    scores = cross_encoder.predict(pairs)

    # スコアの降順で文書を並べ替える
    scored_docs = list(zip(documents, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    # 再ランキング後の上位K件を返す
    return [doc for doc, score in scored_docs[:top_k]]

In [ ]:
def retrieve_with_reranking(
    vectorstore,
    query: str,
    initial_k: int = 20,
    final_k: int = 5
):
    """
    2段階の検索を行う。まず広めに初回検索を行い、その後で精密に再ランキングする。

    Args:
        vectorstore: ChromaDBのベクトルストア
        query (str): 検索クエリ
        initial_k (int): 初回の広範な検索で取得するチャンク数
        final_k (int): 再ランキング後に返すチャンク数

    Returns:
        list: 再ランキングされた上位K件の文書
    """
    # ステージ1：ベクトル類似度を使用して広範に検索する
    initial_results = vectorstore.similarity_search(query, k=initial_k)

    # ステージ2：クロスエンコーダーを使用して精密に再ランキングする
    reranked_results = rerank_with_cross_encoder(
        query,
        initial_results,
        top_k=final_k
    )

    return reranked_results

In [ ]:
def generate_summary_from_reranked_results(
    reranked_results,
    query_topic: str = "risk factors"
):
    """
    再ランキング済みの結果を受け取り、LLMを使用して要約を生成する。

    Args:
        reranked_results (list): retrieve_with_rerankingから得られた、
            再ランキング済みのDocumentオブジェクトのリスト
        query_topic (str): 分析対象のトピック（プロンプトで使用する）

    Returns:
        str: LLMによって生成された要約
    """
    # Documentオブジェクトからテキスト内容を抽出して結合する
    combined_context = "\n\n---CHUNK---\n\n".join(
        [doc.page_content for doc in reranked_results]
    )

    # プロンプトテンプレートを作成する
    prompt_template = """
    ROLE: You are an expert financial analyst specializing in SEC filing
    analysis.

    CONTEXT: Below are the most relevant sections from a 10-K filing related to
    {topic}. These sections were retrieved using semantic search and reranked
    with a cross-encoder to ensure maximum relevance.

    INSTRUCTION: Analyze these sections and create a comprehensive summary of
    {topic}. Organize your analysis into clear categories and highlight the
    most material points.

    RETRIEVED AND RERANKED SECTIONS:
    {context}

    REQUIREMENTS:
    - Organize into logical categories
    - Highlight material/significant points
    - Keep summary concise but comprehensive (under 500 words)
    - Use bullet points for clarity where appropriate

    ANALYSIS:
    """

    prompt = PromptTemplate(
        input_variables=["topic", "context"],
        template=prompt_template
    )

    llm = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)
    chain = LLMChain(llm=llm, prompt=prompt)

    summary = chain.run(topic=query_topic, context=combined_context)

    return summary

# 実行
vectorstore = create_vector_store(ten_k_text)

# ステップ1：検索して再ランキングする
query = """risk factors, threats, vulnerabilities, regulatory concerns,
    market risks"""
reranked_docs = retrieve_with_reranking(
    vectorstore,
    query=query,
    initial_k=20,  # 初回検索では広めに取得する
    final_k=5      # 再ランキング後に上位5件へ絞り込む
)

# ステップ2：再ランキング済みの結果から要約を生成する
risk_summary = generate_summary_from_reranked_results(
    reranked_docs,
    query_topic="risk factors"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
risk_summary

'**Comprehensive Summary of Risk Factors from Apple Inc. 2024 10-K Filing**\n\n---\n\n### 1. **Market and Economic Risks**\n\n- **Interest Rate Risk**\n  - Exposure primarily to fluctuations in U.S. interest rates impacting investment portfolio fair value and term debt interest expense.\n  - A hypothetical 100 basis point increase could reduce investment portfolio fair value by approximately $2.76 billion (2024) and increase annual interest expense on term debt by $139 million.\n  - The Company employs derivatives and portfolio duration management to mitigate these risks but residual impacts remain possible.\n\n- **Foreign Exchange Rate Risk**\n  - The Company is a net receiver of non-U.S. dollar currencies; a strengthening U.S. dollar negatively affects net sales and gross margins in USD terms.\n  - Foreign exchange fluctuations also impact asset and liability valuations.\n  - Hedging strategies include derivatives, offsetting exposures, and local currency pricing adjustments, though 